# 🔄 Autoencoders: Teoría e Implementación Desde Cero

## Aprendiendo a Comprimir y Reconstruir Datos

**Objetivo**: Comprender la teoría, matemáticas e implementación de los Autoencoders y sus variantes:
Vanilla, Sparse, Denoising, Convolucional y Variacional (VAE). Todo implementado desde cero con PyTorch.

### Contenido
1. **Motivación**: ¿Qué es un Autoencoder y para qué sirve?
2. **Arquitectura**: Encoder → Espacio Latente → Decoder
3. **Matemáticas**: Funciones de pérdida, regularización, KL divergence
4. **Ejemplo numérico** paso a paso
5. **Implementación from scratch**:
   - Autoencoder Vanilla
   - Autoencoder Sparse
   - Autoencoder Denoising
   - Autoencoder Convolucional
   - Autoencoder Variacional (VAE)
6. **Experimentos**: MNIST, generación, interpolación en espacio latente

## 1. Motivación: ¿Qué es un Autoencoder?

Un **Autoencoder** es una red neuronal que aprende a **copiar su entrada a su salida**, 
pero con una restricción: debe pasar por una representación intermedia de **menor dimensión** 
(cuello de botella).

```
         AUTOENCODER
    ┌───────────────────────┐
    │                       │
    │  Input   Latente  Output
    │  (784) → (32) →  (784)
    │                       │
    │  ████     ██     ████ │
    │  ████     ██     ████ │
    │  ████  →  ██  →  ████ │
    │  ████     ██     ████ │
    │  ████             ████ │
    │                       │
    │ ENCODER  z    DECODER │
    └───────────────────────┘
    
    Objetivo: x̂ ≈ x  (reconstruir la entrada)
```

### ¿Para qué sirve si solo "copia"?

Lo interesante NO es la copia, sino el **espacio latente** (la representación comprimida):

| Aplicación | Descripción |
|-----------|------------|
| **Reducción de dimensionalidad** | Como PCA pero no-lineal |
| **Compresión de datos** | Representar datos con menos variables |
| **Detección de anomalías** | Datos anómalos se reconstruyen mal |
| **Eliminación de ruido** | Denoising Autoencoders |
| **Generación de datos** | VAE genera nuevos datos realistas |
| **Aprendizaje de features** | Representaciones útiles para otras tareas |
| **Búsqueda por similitud** | Comparar en el espacio latente comprimido |

## 2. Arquitectura del Autoencoder

### 2.1 Componentes

```
    ENTRADA (x)          ESPACIO LATENTE (z)         SALIDA (x̂)
    ┌─────┐               ┌───┐                    ┌─────┐
    │ x₁  │   ┌────────┐  │z₁ │  ┌────────┐       │ x̂₁  │
    │ x₂  │──▶│ENCODER │──│z₂ │──│DECODER │──────▶│ x̂₂  │
    │ x₃  │   │ f(x)   │  │z₃ │  │ g(z)   │       │ x̂₃  │
    │ ...  │   └────────┘  └───┘  └────────┘       │ ...  │
    │ x_n  │                                        │ x̂_n  │
    └─────┘                                        └─────┘
     dim=n            dim=d (d << n)                 dim=n
```

1. **Encoder** $f_\theta$: Comprime la entrada $x$ en una representación latente $z$
   $$z = f_\theta(x) = \sigma(Wx + b)$$

2. **Espacio Latente** $z$: Representación comprimida de dimensión $d \ll n$

3. **Decoder** $g_\phi$: Reconstruye la entrada a partir de $z$
   $$\hat{x} = g_\phi(z) = \sigma'(W'z + b')$$

### 2.2 Tipos de Autoencoders

```
                        AUTOENCODERS
                            │
        ┌───────────┬───────┼───────┬──────────────┐
        │           │       │       │              │
    Vanilla     Sparse  Denoising  Convolucional   VAE
   (básico)   (escasez) (ruido)    (imágenes)  (generativo)
        │           │       │       │              │
    Simple      + L1/KL  + Ruido  + Conv2D    + KL Divergence
    MSE loss    penalty   input    layers     + Reparametrización
                                              + Sampling
```

## 3. Fundamentos Matemáticos

### 3.1 Función de Pérdida Base: Reconstrucción

El objetivo es minimizar la diferencia entre la entrada $x$ y su reconstrucción $\hat{x}$:

**Error Cuadrático Medio (MSE)** — para datos continuos:
$$\mathcal{L}_{\text{MSE}} = \frac{1}{n} \sum_{i=1}^{n} (x_i - \hat{x}_i)^2$$

**Entropía Cruzada Binaria (BCE)** — para datos en $[0, 1]$ (como imágenes normalizadas):
$$\mathcal{L}_{\text{BCE}} = -\frac{1}{n} \sum_{i=1}^{n} \left[ x_i \log(\hat{x}_i) + (1-x_i)\log(1-\hat{x}_i) \right]$$

### 3.2 Autoencoder Sparse: Regularización de Activaciones

Se añade una penalización para que el espacio latente sea **escaso** (pocas neuronas activas):

$$\mathcal{L}_{\text{sparse}} = \mathcal{L}_{\text{recon}} + \lambda \sum_{j=1}^{d} |z_j|$$

O con penalización KL sobre la activación media $\hat{\rho}_j$:
$$\mathcal{L}_{\text{sparse}} = \mathcal{L}_{\text{recon}} + \beta \sum_{j=1}^{d} KL(\rho \| \hat{\rho}_j)$$

Donde $\rho$ es la escasez deseada (ej: 0.05) y $\hat{\rho}_j$ es la activación media de la neurona $j$.

### 3.3 Denoising Autoencoder

Se entrena con entradas **corrompidas** $\tilde{x}$ para reconstruir la entrada **original** $x$:

$$\tilde{x} = x + \epsilon, \quad \epsilon \sim \mathcal{N}(0, \sigma^2)$$
$$\mathcal{L}_{\text{denoise}} = \| x - g_\phi(f_\theta(\tilde{x})) \|^2$$

Esto obliga al modelo a aprender la **estructura subyacente** de los datos, no a memorizar.

### 3.4 Autoencoder Variacional (VAE) — El más importante

El VAE no aprende un punto $z$ en el espacio latente, sino una **distribución**:

$$z \sim q_\theta(z|x) = \mathcal{N}(\mu_\theta(x), \sigma^2_\theta(x))$$

**Encoder**: produce $\mu$ y $\log\sigma^2$ (en vez de un único vector)

**Truco de reparametrización** (para poder hacer backpropagation):
$$z = \mu + \sigma \cdot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

**Función de pérdida del VAE** (ELBO — Evidence Lower Bound):

$$\boxed{\mathcal{L}_{\text{VAE}} = \underbrace{\mathcal{L}_{\text{recon}}(x, \hat{x})}_{\text{Reconstrucción}} + \underbrace{D_{KL}\left(q_\theta(z|x) \| p(z)\right)}_{\text{Regularización}}}$$

Donde el término de **KL Divergence** (con $p(z) = \mathcal{N}(0, I)$):

$$D_{KL} = -\frac{1}{2} \sum_{j=1}^{d} \left(1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

**Interpretación:**
- **Reconstrucción**: que la salida sea parecida a la entrada
- **KL Divergence**: que la distribución latente sea cercana a $\mathcal{N}(0, I)$
  - Esto **regulariza** el espacio latente, haciéndolo continuo y suave
  - Permite **generar** nuevos datos muestreando de $\mathcal{N}(0, I)$

### ¿Por qué $\mathcal{N}(0, I)$ como prior?
- Espacio latente **continuo**: puntos cercanos generan datos similares
- **Interpolación** significativa entre datos
- Fácil de **muestrear** para generación

## 4. Ejemplo Numérico Paso a Paso

Veamos un autoencoder mínimo con datos 2D → 1D → 2D.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

# --- Ejemplo numérico: Autoencoder 2D → 1D → 2D ---
print("=" * 60)
print("EJEMPLO NUMÉRICO: AUTOENCODER MÍNIMO")
print("=" * 60)

# Datos: 4 puntos en 2D
X = np.array([
    [2.0, 1.0],
    [3.0, 2.0],
    [4.0, 3.0],
    [5.0, 4.0],
])
print(f"\nDatos originales X (4×2):\n{X}")

# --- Encoder: 2D → 1D ---
# Pesos (simplificados, normalmente se aprenden)
W_enc = np.array([[0.6], [0.8]])  # (2, 1)
b_enc = np.array([-2.0])          # (1,)

print(f"\n--- ENCODER (2D → 1D) ---")
print(f"W_enc = {W_enc.flatten()}, b_enc = {b_enc}")

# z = ReLU(X · W_enc + b_enc)
z_pre = X @ W_enc + b_enc
z = np.maximum(z_pre, 0)  # ReLU

print(f"\nz_pre = X·W + b:")
for i in range(len(X)):
    print(f"  x={X[i]} → {X[i][0]}×0.6 + {X[i][1]}×0.8 + (-2.0) = {z_pre[i][0]:.2f}")
print(f"\nz (después de ReLU):\n{z.flatten()}")
print(f"  → ¡Los 4 puntos 2D se comprimieron a 1D!")

# --- Decoder: 1D → 2D ---
W_dec = np.array([[0.7, 0.5]])  # (1, 2)
b_dec = np.array([0.5, 0.2])    # (2,)

print(f"\n--- DECODER (1D → 2D) ---")
print(f"W_dec = {W_dec.flatten()}, b_dec = {b_dec}")

# x_hat = z · W_dec + b_dec
x_hat = z @ W_dec + b_dec
print(f"\nx̂ (reconstrucción):\n{x_hat}")

# --- Calcular error ---
mse = np.mean((X - x_hat) ** 2)
print(f"\n--- ERROR ---")
print(f"MSE = {mse:.4f}")
print(f"\nComparación entrada vs reconstrucción:")
for i in range(len(X)):
    error_i = np.sum((X[i] - x_hat[i])**2)
    print(f"  x={X[i]} → x̂={x_hat[i].round(2)} | error²={error_i:.4f}")

# Visualización
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Datos originales
axes[0].scatter(X[:, 0], X[:, 1], c='blue', s=100, zorder=5, label='Original (x)')
axes[0].scatter(x_hat[:, 0], x_hat[:, 1], c='red', s=100, marker='x', zorder=5, label='Reconstruido (x̂)')
for i in range(len(X)):
    axes[0].annotate('', xy=x_hat[i], xytext=X[i],
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
axes[0].set_xlabel('x₁')
axes[0].set_ylabel('x₂')
axes[0].set_title('Original vs Reconstruido', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Espacio latente (1D)
axes[1].scatter(z.flatten(), np.zeros(4), c=['C0','C1','C2','C3'], s=150, zorder=5)
for i in range(4):
    axes[1].annotate(f'x{i+1}→z={z[i][0]:.1f}', (z[i][0], 0.02), fontsize=10, ha='center')
axes[1].set_xlabel('z (dimensión latente)')
axes[1].set_title('Espacio Latente (1D)', fontweight='bold')
axes[1].set_ylim(-0.1, 0.15)
axes[1].axhline(y=0, color='gray', linestyle='-', alpha=0.3)
axes[1].grid(True, alpha=0.3, axis='x')

# 3. Pipeline completo
pipe_y = [2, 1, 0]
labels = ['Entrada (2D)', 'Latente (1D)', 'Salida (2D)']
dims = [2, 1, 2]
colors = ['steelblue', 'orange', 'crimson']
for j, (y, label, dim, c) in enumerate(zip(pipe_y, labels, dims, colors)):
    axes[2].barh(y, dim, color=c, alpha=0.7, height=0.5)
    axes[2].text(dim/2, y, f'{label}\n(dim={dim})', ha='center', va='center', fontsize=10, fontweight='bold')
axes[2].set_xlim(0, 3)
axes[2].set_title('Arquitectura del Autoencoder', fontweight='bold')
axes[2].set_xlabel('Dimensiones')
axes[2].set_yticks([])
axes[2].annotate('', xy=(1.5, 1.3), xytext=(1.5, 1.7), arrowprops=dict(arrowstyle='->', lw=2))
axes[2].annotate('', xy=(1.5, 0.3), xytext=(1.5, 0.7), arrowprops=dict(arrowstyle='->', lw=2))
axes[2].text(2.2, 1.5, 'Encoder', fontsize=11, va='center')
axes[2].text(2.2, 0.5, 'Decoder', fontsize=11, va='center')

plt.tight_layout()
plt.show()

## 5. Implementación From Scratch con PyTorch

Vamos a implementar cada tipo de autoencoder paso a paso, 
entrenándolos en el dataset MNIST (dígitos escritos a mano, 28×28 = 784 píxeles).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import time

# Dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")

# --- Cargar MNIST ---
# Descarga simple sin torchvision
from urllib.request import urlretrieve
import gzip
import os
import struct

def descargar_mnist(directorio='./mnist_data'):
    os.makedirs(directorio, exist_ok=True)
    base_url = 'https://storage.googleapis.com/cvdf-datasets/mnist/'
    archivos = {
        'train_images': 'train-images-idx3-ubyte.gz',
        'train_labels': 'train-labels-idx1-ubyte.gz',
        'test_images': 't10k-images-idx3-ubyte.gz',
        'test_labels': 't10k-labels-idx1-ubyte.gz'
    }
    
    for nombre, archivo in archivos.items():
        ruta = os.path.join(directorio, archivo)
        if not os.path.exists(ruta):
            print(f"Descargando {archivo}...")
            urlretrieve(base_url + archivo, ruta)
    
    def leer_imagenes(ruta):
        with gzip.open(ruta, 'rb') as f:
            magic, n, rows, cols = struct.unpack('>IIII', f.read(16))
            return np.frombuffer(f.read(), dtype=np.uint8).reshape(n, rows * cols)
    
    def leer_labels(ruta):
        with gzip.open(ruta, 'rb') as f:
            magic, n = struct.unpack('>II', f.read(8))
            return np.frombuffer(f.read(), dtype=np.uint8)
    
    X_train = leer_imagenes(os.path.join(directorio, archivos['train_images']))
    y_train = leer_labels(os.path.join(directorio, archivos['train_labels']))
    X_test = leer_imagenes(os.path.join(directorio, archivos['test_images']))
    y_test = leer_labels(os.path.join(directorio, archivos['test_labels']))
    
    return X_train, y_train, X_test, y_test

X_train, y_train, X_test, y_test = descargar_mnist()

# Normalizar a [0, 1]
X_train = X_train.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0

# Crear DataLoaders
train_tensor = torch.FloatTensor(X_train)
test_tensor = torch.FloatTensor(X_test)
train_labels = torch.LongTensor(y_train)
test_labels = torch.LongTensor(y_test)

BATCH_SIZE = 128
train_loader = DataLoader(TensorDataset(train_tensor, train_labels), 
                         batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(test_tensor, test_labels), 
                        batch_size=BATCH_SIZE, shuffle=False)

print(f"\nMNIST cargado:")
print(f"  Train: {X_train.shape} ({X_train.min():.0f} - {X_train.max():.0f})")
print(f"  Test:  {X_test.shape}")
print(f"  Clases: {np.unique(y_train)}")

# Visualizar ejemplos
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i in range(10):
    idx = np.where(y_train == i)[0][0]
    axes[0, i].imshow(X_train[idx].reshape(28, 28), cmap='gray')
    axes[0, i].set_title(f'{i}', fontsize=12)
    axes[0, i].axis('off')
    axes[1, i].imshow(X_train[idx].reshape(28, 28), cmap='hot')
    axes[1, i].axis('off')
plt.suptitle('MNIST: Dígitos 0-9 (784 píxeles cada uno)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.1 Autoencoder Vanilla (Básico)

El autoencoder más simple: capas densas con cuello de botella.

```
Encoder: 784 → 256 → 64 → 32 (espacio latente)
Decoder: 32 → 64 → 256 → 784
```

In [ ]:
class AutoencoderVanilla(nn.Module):
    """
    Autoencoder básico con capas densas.
    
    Arquitectura:
        Encoder: input_dim → 256 → 64 → latent_dim
        Decoder: latent_dim → 64 → 256 → input_dim
    """
    
    def __init__(self, input_dim=784, latent_dim=32):
        super().__init__()
        self.latent_dim = latent_dim
        
        # Encoder: comprime progresivamente
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim),
            nn.ReLU()
        )
        
        # Decoder: expande progresivamente
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 256),
            nn.ReLU(),
            nn.Linear(256, input_dim),
            nn.Sigmoid()  # Salida en [0, 1] (píxeles normalizados)
        )
    
    def encode(self, x):
        return self.encoder(x)
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z

# --- Función de entrenamiento genérica ---
def entrenar_autoencoder(modelo, train_loader, n_epochs=20, lr=1e-3, 
                         noise_factor=0.0, sparse_lambda=0.0, nombre="AE"):
    """Entrena cualquier tipo de autoencoder."""
    optimizer = optim.Adam(modelo.parameters(), lr=lr)
    losses = []
    
    print(f"\nEntrenando {nombre}...")
    start = time.time()
    
    for epoch in range(n_epochs):
        modelo.train()
        epoch_loss = 0
        
        for batch_x, _ in train_loader:
            batch_x = batch_x.to(device)
            
            # Opción: añadir ruido (para Denoising)
            if noise_factor > 0:
                noise = torch.randn_like(batch_x) * noise_factor
                noisy_x = torch.clamp(batch_x + noise, 0., 1.)
                x_hat, z = modelo(noisy_x)
            else:
                x_hat, z = modelo(batch_x)
            
            # Loss de reconstrucción (MSE)
            recon_loss = F.mse_loss(x_hat, batch_x)
            
            # Penalización de escasez L1 (para Sparse)
            sparse_loss = sparse_lambda * torch.mean(torch.abs(z)) if sparse_lambda > 0 else 0
            
            loss = recon_loss + sparse_loss
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(train_loader)
        losses.append(avg_loss)
        
        if (epoch + 1) % 5 == 0:
            print(f"  Época {epoch+1:3d}/{n_epochs} | Loss: {avg_loss:.6f}")
    
    elapsed = time.time() - start
    print(f"  Completado en {elapsed:.1f}s | Loss final: {losses[-1]:.6f}")
    return losses

# --- Función para visualizar reconstrucciones ---
def visualizar_reconstrucciones(modelo, test_data, n=10, titulo="Autoencoder"):
    modelo.eval()
    with torch.no_grad():
        x = test_data[:n].to(device)
        x_hat, z = modelo(x)
        x_hat = x_hat.cpu()
    
    fig, axes = plt.subplots(3, n, figsize=(n * 1.5, 4.5))
    for i in range(n):
        # Original
        axes[0, i].imshow(test_data[i].reshape(28, 28), cmap='gray')
        axes[0, i].axis('off')
        if i == 0: axes[0, i].set_ylabel('Original', fontsize=11)
        
        # Reconstruido
        axes[1, i].imshow(x_hat[i].reshape(28, 28).numpy(), cmap='gray')
        axes[1, i].axis('off')
        if i == 0: axes[1, i].set_ylabel('Reconstruido', fontsize=11)
        
        # Diferencia
        diff = torch.abs(test_data[i] - x_hat[i])
        axes[2, i].imshow(diff.reshape(28, 28).numpy(), cmap='hot')
        axes[2, i].axis('off')
        if i == 0: axes[2, i].set_ylabel('|Diferencia|', fontsize=11)
    
    plt.suptitle(titulo, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# --- Entrenar Autoencoder Vanilla ---
ae_vanilla = AutoencoderVanilla(784, latent_dim=32).to(device)
losses_vanilla = entrenar_autoencoder(ae_vanilla, train_loader, n_epochs=20, nombre="Vanilla AE (dim=32)")

# Visualizar
visualizar_reconstrucciones(ae_vanilla, test_tensor, titulo="Autoencoder Vanilla (latent_dim=32)")

### 5.2 Autoencoder Sparse

Añade una penalización L1 sobre las activaciones latentes para forzar **escasez**: 
que solo unas pocas neuronas se activen para cada entrada.

$$\mathcal{L} = \text{MSE}(x, \hat{x}) + \lambda \cdot \|z\|_1$$

In [ ]:
# --- Entrenar Autoencoder Sparse ---
# Usamos la misma arquitectura pero con penalización L1
ae_sparse = AutoencoderVanilla(784, latent_dim=64).to(device)  # Más neuronas, pero sparse
losses_sparse = entrenar_autoencoder(
    ae_sparse, train_loader, n_epochs=20, 
    sparse_lambda=1e-3,  # Penalización de escasez
    nombre="Sparse AE (dim=64, λ=1e-3)"
)

visualizar_reconstrucciones(ae_sparse, test_tensor, titulo="Autoencoder Sparse (latent_dim=64, λ=1e-3)")

# Comparar activaciones: Vanilla vs Sparse
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

ae_vanilla.eval()
ae_sparse.eval()
with torch.no_grad():
    sample = test_tensor[:200].to(device)
    _, z_van = ae_vanilla(sample)
    _, z_spr = ae_sparse(sample)

# Activaciones promedio
avg_van = z_van.cpu().mean(dim=0).numpy()
avg_spr = z_spr.cpu().mean(dim=0).numpy()

axes[0].bar(range(len(avg_van)), sorted(avg_van, reverse=True), color='steelblue', alpha=0.7)
axes[0].set_title('Vanilla: Activaciones Promedio', fontweight='bold')
axes[0].set_xlabel('Neurona (ordenada)')
axes[0].set_ylabel('Activación media')

axes[1].bar(range(len(avg_spr)), sorted(avg_spr, reverse=True), color='orange', alpha=0.7)
axes[1].set_title('Sparse: Activaciones Promedio', fontweight='bold')
axes[1].set_xlabel('Neurona (ordenada)')
axes[1].set_ylabel('Activación media')

# Histograma de activaciones
axes[2].hist(z_van.cpu().flatten().numpy(), bins=50, alpha=0.6, label='Vanilla', density=True)
axes[2].hist(z_spr.cpu().flatten().numpy(), bins=50, alpha=0.6, label='Sparse', density=True)
axes[2].set_title('Distribución de Activaciones', fontweight='bold')
axes[2].set_xlabel('Valor de activación')
axes[2].legend()

plt.tight_layout()
plt.show()

# Estadísticas
van_sparsity = (z_van.cpu().abs() < 0.1).float().mean().item()
spr_sparsity = (z_spr.cpu().abs() < 0.1).float().mean().item()
print(f"\n📊 Escasez (% neuronas con |z| < 0.1):")
print(f"  Vanilla: {van_sparsity*100:.1f}%")
print(f"  Sparse:  {spr_sparsity*100:.1f}%")
print(f"  → El Sparse AE tiene activaciones más concentradas en cero")

### 5.3 Autoencoder Denoising

Entrena con entradas **corrompidas con ruido** y aprende a reconstruir la entrada **limpia**.
Esto fuerza al modelo a aprender la estructura real de los datos, no memorizar.

```
x (limpio) → añadir ruido → x̃ (ruidoso) → Encoder → z → Decoder → x̂ ≈ x (limpio!)
```

In [ ]:
# --- Entrenar Autoencoder Denoising ---
ae_denoise = AutoencoderVanilla(784, latent_dim=32).to(device)
losses_denoise = entrenar_autoencoder(
    ae_denoise, train_loader, n_epochs=20,
    noise_factor=0.35,  # 35% de ruido gaussiano
    nombre="Denoising AE (ruido=0.35)"
)

# Visualizar capacidad de denoising
ae_denoise.eval()
n = 10
with torch.no_grad():
    x_clean = test_tensor[:n].to(device)
    noise = torch.randn_like(x_clean) * 0.35
    x_noisy = torch.clamp(x_clean + noise, 0., 1.)
    x_denoised, _ = ae_denoise(x_noisy)
    x_denoised = x_denoised.cpu()
    x_noisy = x_noisy.cpu()

fig, axes = plt.subplots(3, n, figsize=(n * 1.5, 4.5))
for i in range(n):
    axes[0, i].imshow(test_tensor[i].reshape(28, 28), cmap='gray')
    axes[0, i].axis('off')
    if i == 0: axes[0, i].set_ylabel('Original', fontsize=11)
    
    axes[1, i].imshow(x_noisy[i].reshape(28, 28), cmap='gray')
    axes[1, i].axis('off')
    if i == 0: axes[1, i].set_ylabel('Con ruido', fontsize=11)
    
    axes[2, i].imshow(x_denoised[i].reshape(28, 28).numpy(), cmap='gray')
    axes[2, i].axis('off')
    if i == 0: axes[2, i].set_ylabel('Denoised', fontsize=11)

plt.suptitle('Denoising Autoencoder: Eliminación de Ruido', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# MSE comparison
mse_noisy = F.mse_loss(x_noisy, test_tensor[:n]).item()
mse_denoised = F.mse_loss(x_denoised, test_tensor[:n]).item()
print(f"\n📊 Error de reconstrucción:")
print(f"  Ruidoso vs Original: MSE = {mse_noisy:.4f}")
print(f"  Denoised vs Original: MSE = {mse_denoised:.4f}")
print(f"  Reducción de error: {(1 - mse_denoised/mse_noisy)*100:.1f}%")

### 5.4 Autoencoder Convolucional

Para imágenes, las capas convolucionales son más eficientes que las densas:
- Capturan **patrones espaciales** (bordes, texturas)
- Tienen **menos parámetros** (pesos compartidos)
- Preservan la **estructura 2D** de la imagen

```
Encoder: (1,28,28) → Conv → Conv → Flatten → latent
Decoder: latent → Unflatten → ConvT → ConvT → (1,28,28)
```

In [ ]:
class AutoencoderConvolucional(nn.Module):
    """
    Autoencoder con capas convolucionales para imágenes.
    
    Encoder: Conv2d + MaxPool reducen dimensionalidad espacial
    Decoder: ConvTranspose2d (deconv) reconstruyen la imagen
    """
    
    def __init__(self, latent_dim=16):
        super().__init__()
        self.latent_dim = latent_dim
        
        # Encoder convolucional
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),  # (1,28,28) → (16,14,14)
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),  # (16,14,14) → (32,7,7)
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),  # (32,7,7) → (64,4,4)
            nn.ReLU(),
        )
        self.encoder_fc = nn.Linear(64 * 4 * 4, latent_dim)
        
        # Decoder convolucional (transpuesto)
        self.decoder_fc = nn.Linear(latent_dim, 64 * 4 * 4)
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=0),  # (64,4,4) → (32,7,7)
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),  # (32,7,7) → (16,14,14)
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1),  # (16,14,14) → (1,28,28)
            nn.Sigmoid(),
        )
    
    def encode(self, x):
        x = x.view(-1, 1, 28, 28)
        h = self.encoder_conv(x)
        h = h.view(h.size(0), -1)  # flatten
        z = self.encoder_fc(h)
        return z
    
    def decode(self, z):
        h = self.decoder_fc(z)
        h = h.view(-1, 64, 4, 4)
        x_hat = self.decoder_conv(h)
        return x_hat.view(-1, 784)
    
    def forward(self, x):
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z

# Entrenar
ae_conv = AutoencoderConvolucional(latent_dim=16).to(device)
losses_conv = entrenar_autoencoder(ae_conv, train_loader, n_epochs=20, nombre="Conv AE (dim=16)")

# Visualizar
visualizar_reconstrucciones(ae_conv, test_tensor, titulo="Autoencoder Convolucional (latent_dim=16)")

# Comparar parámetros
params_vanilla = sum(p.numel() for p in ae_vanilla.parameters())
params_conv = sum(p.numel() for p in ae_conv.parameters())
print(f"\n📊 Comparación de parámetros:")
print(f"  Vanilla (dim=32): {params_vanilla:,} parámetros")
print(f"  Conv    (dim=16): {params_conv:,} parámetros")
print(f"  Conv tiene {params_conv/params_vanilla*100:.0f}% de los parámetros del Vanilla")
print(f"  ¡Y con MENOR dimensión latente (16 vs 32)!")

### 5.5 Autoencoder Variacional (VAE)

El VAE es el tipo más poderoso y teóricamente elegante. En lugar de aprender
un punto fijo en el espacio latente, aprende una **distribución**.

```
         VAE
    ┌──────────────────────────────────────────┐
    │                                          │
    │  Input → Encoder → μ, log(σ²)           │
    │                       ↓                  │
    │              z = μ + σ·ε   (ε ~ N(0,1))  │
    │                       ↓                  │
    │              Decoder → Output             │
    │                                          │
    │  Loss = Reconstrucción + KL(q(z|x)||p(z))│
    └──────────────────────────────────────────┘
```

**Clave**: El truco de reparametrización permite hacer backpropagation a través del muestreo.

In [ ]:
class VAE(nn.Module):
    """
    Autoencoder Variacional (VAE).
    
    Diferencias con un AE normal:
    1. El encoder produce μ y log(σ²) en vez de un solo z
    2. z se muestrea usando el truco de reparametrización
    3. La pérdida incluye KL divergence
    """
    
    def __init__(self, input_dim=784, hidden_dim=256, latent_dim=2):
        super().__init__()
        self.latent_dim = latent_dim
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
        )
        
        # Capas para μ y log(σ²) — DOS salidas separadas
        self.fc_mu = nn.Linear(128, latent_dim)       # Media
        self.fc_logvar = nn.Linear(128, latent_dim)    # Log-varianza
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()
        )
    
    def encode(self, x):
        """Encoder: x → (μ, log σ²)"""
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar
    
    def reparametrize(self, mu, logvar):
        """
        Truco de reparametrización:
        z = μ + σ · ε,  donde ε ~ N(0, I)
        
        Esto permite que el gradiente fluya a través del muestreo.
        Sin este truco, no se puede entrenar con backpropagation.
        """
        std = torch.exp(0.5 * logvar)  # σ = exp(log(σ²)/2)
        eps = torch.randn_like(std)     # ε ~ N(0, I)
        z = mu + std * eps              # z = μ + σ · ε
        return z
    
    def decode(self, z):
        """Decoder: z → x̂"""
        return self.decoder(z)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparametrize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar, z
    
    @staticmethod
    def loss_function(x, x_hat, mu, logvar, beta=1.0):
        """
        Pérdida del VAE = Reconstrucción + β · KL Divergence
        
        KL(q(z|x) || p(z)) = -½ Σ(1 + log(σ²) - μ² - σ²)
        
        Args:
            beta: peso del término KL (β-VAE cuando β > 1)
        """
        # Reconstrucción (BCE)
        recon_loss = F.binary_cross_entropy(x_hat, x, reduction='sum')
        
        # KL Divergence
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        
        return recon_loss + beta * kl_loss, recon_loss, kl_loss

# --- Entrenamiento especial para VAE ---
def entrenar_vae(modelo, train_loader, n_epochs=30, lr=1e-3, beta=1.0):
    optimizer = optim.Adam(modelo.parameters(), lr=lr)
    losses_total, losses_recon, losses_kl = [], [], []
    
    print(f"\nEntrenando VAE (latent_dim={modelo.latent_dim}, β={beta})...")
    start = time.time()
    
    for epoch in range(n_epochs):
        modelo.train()
        total, recon_sum, kl_sum = 0, 0, 0
        
        for batch_x, _ in train_loader:
            batch_x = batch_x.to(device)
            x_hat, mu, logvar, z = modelo(batch_x)
            
            loss, recon, kl = VAE.loss_function(batch_x, x_hat, mu, logvar, beta)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total += loss.item()
            recon_sum += recon.item()
            kl_sum += kl.item()
        
        n = len(train_loader.dataset)
        losses_total.append(total / n)
        losses_recon.append(recon_sum / n)
        losses_kl.append(kl_sum / n)
        
        if (epoch + 1) % 5 == 0:
            print(f"  Época {epoch+1:3d}/{n_epochs} | Total: {total/n:.2f} | "
                  f"Recon: {recon_sum/n:.2f} | KL: {kl_sum/n:.2f}")
    
    print(f"  Completado en {time.time()-start:.1f}s")
    return losses_total, losses_recon, losses_kl

# Entrenar VAE con 2 dimensiones latentes (para visualización)
vae_2d = VAE(input_dim=784, hidden_dim=256, latent_dim=2).to(device)
losses_vae, losses_r, losses_k = entrenar_vae(vae_2d, train_loader, n_epochs=30, beta=1.0)

In [ ]:
# --- Visualizar el espacio latente 2D del VAE ---
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Espacio latente coloreado por dígito
vae_2d.eval()
all_z = []
all_labels = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        _, mu, _, _ = vae_2d(batch_x.to(device))
        all_z.append(mu.cpu())
        all_labels.append(batch_y)

all_z = torch.cat(all_z, dim=0).numpy()
all_labels = torch.cat(all_labels, dim=0).numpy()

scatter = axes[0].scatter(all_z[:, 0], all_z[:, 1], c=all_labels, 
                          cmap='tab10', s=1, alpha=0.5)
axes[0].set_xlabel('z₁ (μ₁)')
axes[0].set_ylabel('z₂ (μ₂)')
axes[0].set_title('Espacio Latente (μ) por Dígito', fontsize=13, fontweight='bold')
plt.colorbar(scatter, ax=axes[0], ticks=range(10))

# 2. Curvas de pérdida
axes[1].plot(losses_r, label='Reconstrucción', linewidth=2)
axes[1].plot(losses_k, label='KL Divergence', linewidth=2)
axes[1].plot(losses_vae, label='Total', linewidth=2, linestyle='--')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Pérdida')
axes[1].set_title('Curvas de Pérdida del VAE', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. Generar dígitos muestreando del espacio latente
n_grid = 15
grid_x = np.linspace(-3, 3, n_grid)
grid_y = np.linspace(-3, 3, n_grid)[::-1]  # Invertir Y

canvas = np.zeros((28 * n_grid, 28 * n_grid))

with torch.no_grad():
    for i, yi in enumerate(grid_y):
        for j, xi in enumerate(grid_x):
            z_sample = torch.FloatTensor([[xi, yi]]).to(device)
            x_gen = vae_2d.decode(z_sample).cpu().numpy()
            canvas[i*28:(i+1)*28, j*28:(j+1)*28] = x_gen.reshape(28, 28)

axes[2].imshow(canvas, cmap='gray')
axes[2].set_title('Generación: Muestreo del Espacio Latente', fontsize=13, fontweight='bold')
axes[2].set_xlabel('z₁')
axes[2].set_ylabel('z₂')
# Ajustar ticks
tick_positions = np.linspace(14, 28*n_grid-14, 5)
tick_labels = [f'{v:.1f}' for v in np.linspace(-3, 3, 5)]
axes[2].set_xticks(tick_positions)
axes[2].set_xticklabels(tick_labels)
tick_labels_y = [f'{v:.1f}' for v in np.linspace(3, -3, 5)]
axes[2].set_yticks(tick_positions)
axes[2].set_yticklabels(tick_labels_y)

plt.tight_layout()
plt.show()

print("\n💡 Observaciones del VAE 2D:")
print("   - Los dígitos se organizan en clusters suaves en el espacio latente")
print("   - Dígitos similares (3, 5, 8) están cerca en el espacio")
print("   - Se puede GENERAR nuevos dígitos muestreando z ~ N(0, I)")
print("   - Las transiciones entre dígitos son suaves y continuas")

### 5.6 Interpolación en el Espacio Latente del VAE

Una propiedad fascinante del VAE: podemos **interpolar** suavemente entre dos datos.

In [ ]:
# --- Interpolación entre dígitos ---
fig, axes_grid = plt.subplots(5, 12, figsize=(16, 7))

vae_2d.eval()

# Pares de dígitos para interpolar
pares = [(0, 1), (3, 8), (4, 9), (2, 7), (6, 5)]

for row, (dig_a, dig_b) in enumerate(pares):
    # Encontrar un ejemplo de cada dígito
    idx_a = np.where(y_test == dig_a)[0][0]
    idx_b = np.where(y_test == dig_b)[0][0]
    
    xa = test_tensor[idx_a:idx_a+1].to(device)
    xb = test_tensor[idx_b:idx_b+1].to(device)
    
    # Codificar
    with torch.no_grad():
        mu_a, _ = vae_2d.encode(xa)
        mu_b, _ = vae_2d.encode(xb)
    
    # Interpolar linealmente en el espacio latente
    n_steps = 12
    for i in range(n_steps):
        alpha = i / (n_steps - 1)
        z_interp = (1 - alpha) * mu_a + alpha * mu_b
        
        with torch.no_grad():
            x_gen = vae_2d.decode(z_interp).cpu().numpy()
        
        axes_grid[row, i].imshow(x_gen.reshape(28, 28), cmap='gray')
        axes_grid[row, i].axis('off')
        
        if i == 0:
            axes_grid[row, i].set_title(f'{dig_a}', fontsize=11, color='blue')
        elif i == n_steps - 1:
            axes_grid[row, i].set_title(f'{dig_b}', fontsize=11, color='red')

plt.suptitle('Interpolación Lineal en el Espacio Latente del VAE', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("✅ La transición entre dígitos es suave y continua")
print("   Esto demuestra que el VAE aprende un espacio latente ESTRUCTURADO")

### 5.7 VAE con Mayor Dimensión Latente + Generación

Con más dimensiones latentes, la reconstrucción mejora significativamente.

In [ ]:
# VAE con más dimensiones
vae_20d = VAE(input_dim=784, hidden_dim=512, latent_dim=20).to(device)
losses_20d, _, _ = entrenar_vae(vae_20d, train_loader, n_epochs=30, beta=1.0)

# Comparar reconstrucción: VAE 2D vs 20D
fig, axes = plt.subplots(4, 10, figsize=(15, 6))

vae_2d.eval()
vae_20d.eval()
n = 10

with torch.no_grad():
    x = test_tensor[:n].to(device)
    x_hat_2d, _, _, _ = vae_2d(x)
    x_hat_20d, _, _, _ = vae_20d(x)

labels_row = ['Original', 'VAE (2D)', 'VAE (20D)', '|Dif 20D|']
for i in range(n):
    axes[0, i].imshow(test_tensor[i].reshape(28, 28), cmap='gray')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(x_hat_2d[i].cpu().reshape(28, 28).numpy(), cmap='gray')
    axes[1, i].axis('off')
    
    axes[2, i].imshow(x_hat_20d[i].cpu().reshape(28, 28).numpy(), cmap='gray')
    axes[2, i].axis('off')
    
    diff = torch.abs(test_tensor[i] - x_hat_20d[i].cpu())
    axes[3, i].imshow(diff.reshape(28, 28).numpy(), cmap='hot')
    axes[3, i].axis('off')

for i, label in enumerate(labels_row):
    axes[i, 0].set_ylabel(label, fontsize=11)

plt.suptitle('Comparación: VAE 2D vs 20D', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Generar dígitos nuevos (muestrear de N(0,I))
fig, axes = plt.subplots(3, 10, figsize=(15, 4.5))
fig.suptitle('Dígitos GENERADOS por el VAE (20D) — muestreando z ~ N(0, I)', 
             fontsize=14, fontweight='bold')

with torch.no_grad():
    for row in range(3):
        z_random = torch.randn(10, 20).to(device)
        generated = vae_20d.decode(z_random).cpu().numpy()
        for i in range(10):
            axes[row, i].imshow(generated[i].reshape(28, 28), cmap='gray')
            axes[row, i].axis('off')

plt.tight_layout()
plt.show()

print("\n📊 MSE de reconstrucción en test:")
with torch.no_grad():
    x_test = test_tensor.to(device)
    xh_2d, _, _, _ = vae_2d(x_test)
    xh_20d, _, _, _ = vae_20d(x_test)
    mse_2d = F.mse_loss(xh_2d.cpu(), test_tensor).item()
    mse_20d = F.mse_loss(xh_20d.cpu(), test_tensor).item()

print(f"  VAE  2D: {mse_2d:.4f}")
print(f"  VAE 20D: {mse_20d:.4f}")
print(f"  Mejora: {(1 - mse_20d/mse_2d)*100:.1f}%")

## 6. Aplicación: Detección de Anomalías

Un autoencoder entrenado en datos "normales" reconstruye mal los datos **anómalos**.
La idea: si el error de reconstrucción es alto, el dato es una anomalía.

In [ ]:
# --- Detección de anomalías ---
# Entrenar un AE solo con dígitos 0-4 (normales)
# Los dígitos 5-9 serán "anomalías"

# Preparar datos
mask_normal = np.isin(y_train, [0, 1, 2, 3, 4])
X_normal = X_train[mask_normal]
normal_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_normal), torch.zeros(len(X_normal)).long()),
    batch_size=128, shuffle=True
)

ae_anomaly = AutoencoderVanilla(784, latent_dim=20).to(device)
_ = entrenar_autoencoder(ae_anomaly, normal_loader, n_epochs=15, nombre="AE Anomaly Detector")

# Calcular errores de reconstrucción
ae_anomaly.eval()
with torch.no_grad():
    x_test_all = test_tensor.to(device)
    x_hat_all, _ = ae_anomaly(x_test_all)
    errors = ((x_test_all - x_hat_all) ** 2).mean(dim=1).cpu().numpy()

# Separar normales vs anomalías
mask_normal_test = np.isin(y_test, [0, 1, 2, 3, 4])
errors_normal = errors[mask_normal_test]
errors_anomaly = errors[~mask_normal_test]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Distribución de errores
axes[0].hist(errors_normal, bins=50, alpha=0.7, label='Normal (0-4)', density=True, color='steelblue')
axes[0].hist(errors_anomaly, bins=50, alpha=0.7, label='Anomalía (5-9)', density=True, color='crimson')
axes[0].axvline(x=np.percentile(errors_normal, 95), color='black', linestyle='--', label='Umbral (P95)')
axes[0].set_xlabel('Error de reconstrucción (MSE)')
axes[0].set_ylabel('Densidad')
axes[0].set_title('Distribución de Errores', fontweight='bold')
axes[0].legend()

# 2. Error promedio por dígito
errors_by_digit = {}
for d in range(10):
    mask_d = y_test == d
    errors_by_digit[d] = errors[mask_d].mean()

colors = ['steelblue' if d < 5 else 'crimson' for d in range(10)]
axes[1].bar(range(10), [errors_by_digit[d] for d in range(10)], color=colors, alpha=0.8)
axes[1].set_xlabel('Dígito')
axes[1].set_ylabel('Error MSE promedio')
axes[1].set_title('Error Promedio por Dígito', fontweight='bold')
axes[1].set_xticks(range(10))

# 3. Ejemplos de detección
threshold = np.percentile(errors_normal, 95)
# Top anomalías y top normales
top_anomaly_idx = np.argsort(errors)[-5:][::-1]
top_normal_idx = np.argsort(errors)[:5]

for i, (idx, label) in enumerate(
    [(top_normal_idx[j], 'Normal') for j in range(5)] + 
    [(top_anomaly_idx[j], 'Anomalía') for j in range(5)]
):
    row = 0 if i < 5 else 1
    col = i % 5
    
    if i == 0:
        ax_sub = axes[2].inset_axes([col*0.19, 0.55, 0.17, 0.4])
    elif i < 5:
        ax_sub = axes[2].inset_axes([col*0.19, 0.55, 0.17, 0.4])
    else:
        ax_sub = axes[2].inset_axes([col*0.19, 0.05, 0.17, 0.4])
    
    ax_sub.imshow(X_test[idx].reshape(28, 28), cmap='gray')
    ax_sub.set_title(f'{y_test[idx]}:{errors[idx]:.4f}', fontsize=8)
    ax_sub.axis('off')

axes[2].set_title('Normales (arriba) vs Anomalías (abajo)', fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()

# Calcular accuracy de detección
y_pred_anomaly = errors > threshold
y_true_anomaly = ~mask_normal_test
tp = np.sum(y_pred_anomaly & y_true_anomaly)
fp = np.sum(y_pred_anomaly & ~y_true_anomaly)
fn = np.sum(~y_pred_anomaly & y_true_anomaly)
tn = np.sum(~y_pred_anomaly & ~y_true_anomaly)

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"\n📊 Detección de Anomalías (umbral = P95 de normales):")
print(f"  Precision: {precision:.3f}")
print(f"  Recall:    {recall:.3f}")
print(f"  F1-Score:  {f1:.3f}")
print(f"\n  Los dígitos 5-9 se detectan como anomalías porque el AE")
print(f"  nunca los vio durante el entrenamiento → reconstrucción pobre")

## 7. Experimento: Efecto de la Dimensión Latente

¿Qué pasa cuando variamos el tamaño del cuello de botella?

In [ ]:
# --- Comparar dimensiones latentes ---
dims = [2, 4, 8, 16, 32, 64]
resultados = {}

for dim in dims:
    ae = AutoencoderVanilla(784, latent_dim=dim).to(device)
    losses = entrenar_autoencoder(ae, train_loader, n_epochs=15, 
                                  nombre=f"AE dim={dim}")
    
    # Evaluar MSE en test
    ae.eval()
    with torch.no_grad():
        x_hat, _ = ae(test_tensor.to(device))
        mse = F.mse_loss(x_hat.cpu(), test_tensor).item()
    
    resultados[dim] = {'model': ae, 'losses': losses, 'mse': mse}

# Visualización
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# 1. MSE vs dimensión
mses = [resultados[d]['mse'] for d in dims]
axes[0].plot(dims, mses, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Dimensión Latente', fontsize=12)
axes[0].set_ylabel('MSE (Test)', fontsize=12)
axes[0].set_title('Error de Reconstrucción vs Dimensión Latente', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
for d, m in zip(dims, mses):
    axes[0].annotate(f'{m:.4f}', (d, m), textcoords="offset points", 
                    xytext=(0, 10), fontsize=9, ha='center')

# 2. Reconstrucciones para cada dimensión
n_show = 5
sample_idx = [0, 3, 5, 7, 9]

# Original
for j, si in enumerate(sample_idx):
    ax = axes[1].inset_axes([j*0.19 + 0.02, 0.78, 0.15, 0.2])
    ax.imshow(test_tensor[si].reshape(28, 28), cmap='gray')
    ax.axis('off')
    if j == 0: ax.set_title('Original', fontsize=8)

for row_idx, dim in enumerate(dims):
    model = resultados[dim]['model']
    model.eval()
    with torch.no_grad():
        for j, si in enumerate(sample_idx):
            x_hat, _ = model(test_tensor[si:si+1].to(device))
            y_pos = 0.65 - row_idx * 0.12
            ax = axes[1].inset_axes([j*0.19 + 0.02, y_pos, 0.15, 0.2])
            ax.imshow(x_hat[0].cpu().reshape(28, 28).numpy(), cmap='gray')
            ax.axis('off')
            if j == 0: ax.set_title(f'd={dim}', fontsize=8)

axes[1].axis('off')
axes[1].set_title('Reconstrucción por Dimensión Latente', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n📊 Resumen:")
for dim in dims:
    print(f"  dim={dim:3d} → MSE: {resultados[dim]['mse']:.4f}")
print(f"\n💡 Mayor dimensión = mejor reconstrucción, pero:")
print(f"   - Menos compresión")
print(f"   - Riesgo de sobreajuste (memorizar en vez de generalizar)")
print(f"   - Para generación (VAE), dims bajas dan mejor estructura")

## 8. Comparación Final de Todos los Autoencoders

In [ ]:
# --- Comparación final ---
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Recolectar todos los losses
all_losses = {
    'Vanilla (32)': losses_vanilla,
    'Sparse (64)': losses_sparse,
    'Denoising (32)': losses_denoise,
    'Conv (16)': losses_conv,
}

# 1. Curvas de pérdida
for name, losses in all_losses.items():
    axes[0, 0].plot(losses, linewidth=2, label=name)
axes[0, 0].set_xlabel('Época')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Curvas de Entrenamiento', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. MSE en test para cada modelo
modelos = {
    'Vanilla': ae_vanilla,
    'Sparse': ae_sparse,
    'Denoising': ae_denoise,
    'Conv': ae_conv,
}

mse_test = {}
for name, model in modelos.items():
    model.eval()
    with torch.no_grad():
        x_hat, _ = model(test_tensor.to(device))
        mse_test[name] = F.mse_loss(x_hat.cpu(), test_tensor).item()

colors = ['steelblue', 'orange', 'green', 'crimson']
bars = axes[0, 1].bar(mse_test.keys(), mse_test.values(), color=colors, alpha=0.8)
axes[0, 1].set_ylabel('MSE (Test)')
axes[0, 1].set_title('Error de Reconstrucción en Test', fontweight='bold')
for bar, val in zip(bars, mse_test.values()):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                   f'{val:.4f}', ha='center', fontsize=10)

# 3. Número de parámetros
params = {name: sum(p.numel() for p in model.parameters()) for name, model in modelos.items()}
axes[0, 2].barh(list(params.keys()), list(params.values()), color=colors, alpha=0.8)
axes[0, 2].set_xlabel('Parámetros')
axes[0, 2].set_title('Número de Parámetros', fontweight='bold')
for i, (name, p) in enumerate(params.items()):
    axes[0, 2].text(p + 1000, i, f'{p:,}', va='center', fontsize=10)

# 4-6. Reconstrucciones comparadas
n = 8
sample = test_tensor[:n].to(device)

# Original
for i in range(n):
    ax = axes[1, 0].inset_axes([i/n, 0.75, 1/n - 0.01, 0.22])
    ax.imshow(test_tensor[i].reshape(28, 28), cmap='gray')
    ax.axis('off')
    if i == 0: ax.set_ylabel('Orig', fontsize=8)

for row_idx, (name, model) in enumerate(modelos.items()):
    model.eval()
    with torch.no_grad():
        x_hat, _ = model(sample)
    for i in range(n):
        y = 0.52 - row_idx * 0.2
        ax = axes[1, 0].inset_axes([i/n, y, 1/n - 0.01, 0.22])
        ax.imshow(x_hat[i].cpu().reshape(28, 28).numpy(), cmap='gray')
        ax.axis('off')
        if i == 0: ax.set_ylabel(name, fontsize=8)

axes[1, 0].axis('off')
axes[1, 0].set_title('Reconstrucciones Comparadas', fontweight='bold')

# 5. VAE losses
axes[1, 1].plot(losses_r, label='Reconstrucción', linewidth=2)
axes[1, 1].plot(losses_k, label='KL Divergence', linewidth=2)
axes[1, 1].set_xlabel('Época')
axes[1, 1].set_ylabel('Pérdida')
axes[1, 1].set_title('VAE: Descomposición de la Pérdida', fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# 6. Tabla resumen
resumen = [
    ['Tipo', 'Dim Lat', 'MSE', 'Params', 'Especial'],
    ['Vanilla', '32', f"{mse_test['Vanilla']:.4f}", f"{params['Vanilla']:,}", 'Base'],
    ['Sparse', '64', f"{mse_test['Sparse']:.4f}", f"{params['Sparse']:,}", '+ L1 penalty'],
    ['Denoising', '32', f"{mse_test['Denoising']:.4f}", f"{params['Denoising']:,}", '+ Ruido'],
    ['Conv', '16', f"{mse_test['Conv']:.4f}", f"{params['Conv']:,}", 'Conv2D'],
    ['VAE 2D', '2', f'{mse_2d:.4f}', '-', '+ KL div'],
    ['VAE 20D', '20', f'{mse_20d:.4f}', '-', '+ KL div'],
]

axes[1, 2].axis('off')
table = axes[1, 2].table(
    cellText=resumen[1:], colLabels=resumen[0],
    loc='center', cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.5)
for i in range(len(resumen[0])):
    table[0, i].set_facecolor('#4472C4')
    table[0, i].set_text_props(color='white', fontweight='bold')
axes[1, 2].set_title('Resumen de Modelos', fontweight='bold')

plt.tight_layout()
plt.show()

## 9. Resumen y Conclusiones

### Lo que aprendimos:

| Autoencoder | Pérdida | Uso Principal |
|------------|---------|---------------|
| **Vanilla** | $\text{MSE}(x, \hat{x})$ | Compresión, reducción de dimensionalidad |
| **Sparse** | $\text{MSE} + \lambda\|z\|_1$ | Features escasas, selección de características |
| **Denoising** | $\text{MSE}(x, g(f(x+\epsilon)))$ | Eliminación de ruido, representaciones robustas |
| **Convolucional** | $\text{MSE}(x, \hat{x})$ | Imágenes (menos parámetros, estructura 2D) |
| **VAE** | $\text{Recon} + \text{KL}(q\|p)$ | **Generación** de datos nuevos, interpolación |

### Ideas clave:

1. **Cuello de botella** → fuerza al modelo a aprender lo esencial
2. **Espacio latente** → representación comprimida y útil de los datos
3. **VAE** → espacio latente **continuo y estructurado**, permite generación
4. **Detección de anomalías** → datos raros se reconstruyen mal
5. **Trade-off dimensión**: más dims = mejor reconstrucción, menos compresión

### Relación con otros modelos:

```
Autoencoders → Fundamento para:
├── VAE → Modelos generativos (DALL-E, Stable Diffusion usa VAE como componente)
├── GAN → Adversarial training (discriminador vs generador)
├── Diffusion Models → Denoising iterativo en espacio latente
└── Representation Learning → Transfer learning, embeddings
```

---
*Notebook creado como parte de la serie "ML from Scratch"*